# BayMorph — multi-galaxy analysis

Run a shared configuration template over multiple FITS images. The notebook records failures instead of discarding completed objects and writes a checkpoint table after every target.

## Setup

In [ ]:
from pathlib import Path
import copy
import os
import traceback

import pandas as pd

from baymorph import BayMorphRunner, load_resolved_config

if Path.cwd().name == "notebooks":
    os.chdir("..")

## User settings

In [ ]:
BASE_CONFIG = Path("configs/real_galaxy.yaml")
DATA_DIR = Path("data")
PATTERN = "*.fits"
BAND = None
MAX_OBJECTS = None
RUN_DIAGNOSTICS = True
CONTINUE_ON_ERROR = True

# This simple workflow assumes the same target_position is appropriate for all
# images. For non-aligned data, update target_position per object from a catalog.

## Discover input images

In [ ]:
fits_files = sorted(DATA_DIR.glob(PATTERN))

if MAX_OBJECTS is not None:
    fits_files = fits_files[:MAX_OBJECTS]

if not fits_files:
    raise FileNotFoundError(
        f"No FITS files matched {DATA_DIR / PATTERN}."
    )

print(f"Found {len(fits_files)} FITS files.")
fits_files

## Run BayMorph

In [ ]:
base_config = load_resolved_config(BASE_CONFIG)
summary_path = (
    Path(base_config["outputs"]["root"])
    / "multi_galaxy_summary.csv"
)
summary_path.parent.mkdir(parents=True, exist_ok=True)

results = []

for index, image_path in enumerate(fits_files, start=1):
    config = copy.deepcopy(base_config)
    object_name = image_path.stem

    config["object"]["name"] = object_name
    config["data"]["image_path"] = str(image_path)

    print("\n" + "=" * 72)
    print(f"[{index}/{len(fits_files)}] {object_name}")
    print(image_path)
    print("=" * 72)

    try:
        runner = BayMorphRunner.from_config(config)
        runner.prepare_image()

        if config["psf"]["type"] == "empirical":
            runner.select_psf_candidates()
            runner.build_psf()

        fit_products = runner.fit(return_result=True)

        if RUN_DIAGNOSTICS:
            runner.diagnostics()

        products_by_model = (
            fit_products
            if isinstance(fit_products, dict)
            else {fit_products.model_name: fit_products}
        )

        for model_name, product in products_by_model.items():
            posterior = product.posterior
            logz = None if posterior is None else float(posterior.logz[-1])
            logzerr = (
                None
                if posterior is None or posterior.logzerr is None
                else float(posterior.logzerr[-1])
            )

            results.append({
                "object": object_name,
                "band": BAND,
                "image_path": str(image_path),
                "model": model_name,
                "status": "completed",
                "chi2": product.chi2,
                "reduced_chi2": product.reduced_chi2,
                "ndof": product.ndof,
                "logz": logz,
                "logzerr": logzerr,
                "error": None,
            })

    except Exception as exc:
        traceback.print_exc()
        results.append({
            "object": object_name,
            "band": BAND,
            "image_path": str(image_path),
            "model": None,
            "status": "failed",
            "chi2": None,
            "reduced_chi2": None,
            "ndof": None,
            "logz": None,
            "logzerr": None,
            "error": f"{type(exc).__name__}: {exc}",
        })

        if not CONTINUE_ON_ERROR:
            raise

    pd.DataFrame(results).to_csv(summary_path, index=False)
    print("Checkpoint saved:", summary_path)

## Summary

In [ ]:
summary = pd.DataFrame(results)
summary

In [ ]:
print("Saved:", summary_path)
print(summary["status"].value_counts(dropna=False))